# What Information in Encrypted Traffic Actually Helps Classification?

**Alessio Olivieri — University project**

This notebook examines saved results from a PyTorch Geometric adaptation of
TFE-GNN. It reads compact experiment outputs and runs no training.

## Research question

Does the benefit of packet payloads come from encrypted byte values themselves?
Full transport payloads can also contain cleartext, protocol framing and
handshake information. A full-payload intervention alone cannot isolate ciphertext.

Architecture reference: Zhang et al. (2023), *TFE-GNN: A Temporal Fusion Encoder
Using Graph Neural Networks for Fine-grained Encrypted Traffic Classification*,
WWW, pp. 2066–2075. [Official implementation](https://github.com/ViktorAxelsen/TFE-GNN).


## Architecture and data

Header and payload bytes form separate packet-local positive-PMI graphs
(window 5). Each encoder embeds byte values and applies four mean-GraphSAGE
layers, concatenating the layer readouts. Cross-gated fusion feeds a two-layer
bidirectional LSTM and a six-class classifier. The canonical model preserves the
validated GraphSAGE initialization, input dropout and PReLU → BatchNorm ordering.

The study uses 31 ISCXVPN2016 captures and 1,674 eligible bidirectional IPv4/TCP
flows: Chat, Email, FileTransfer, P2P, Streaming and VoIP. Keep the first 50
headers and independently first 50 nonempty payloads, cap at 40/150 bytes, and
pad with token 256. Remove IP addresses and TCP ports; preserve other header
fields. Retransmissions remain; no idle timeout is applied. Empty-payload flows
and flows with more than 10,000 nonempty payload packets are excluded.

The saved split has 1,172/252/250 train/validation/test samples. Endpoint tuples
and identical representations are grouped transitively across captures. It is
flow-disjoint and shares capture sources. Class labels come from capture folders.


In [ ]:
from pathlib import Path
import csv
import json
try:
    from IPython.display import Markdown, display
except ImportError:  # Also support script-based checks without a notebook frontend.
    Markdown, display = str, print

ROOT = Path.cwd()
if not (ROOT / "results").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "results/payload_ablation/summary.csv").is_file(), "Open from the repository or notebooks directory"

def read_csv(relative):
    with (ROOT / relative).open() as handle:
        return list(csv.DictReader(handle))

payload = read_csv("results/payload_ablation/summary.csv")
tls = read_csv("results/tls_ciphertext_ablation/summary.csv")


## Full-payload ablation

Compare captured payload, uniform original-length random payload, and header-only
inputs. Randomization precedes truncation, padding and graph construction;
headers, lengths, order, labels and split membership stay fixed. Header-only
bypasses the payload encoder and sends zeros to fusion, removing that branch's
length information too.

Three paired training seeds (32, 42, 52) share split seed 32 and intervention seed
32. Each run uses 20 epochs, Adam LR .01 with warmup/cosine decay, minibatch 8 and
effective batch 32. Validation macro F1 selects the checkpoint, with validation
loss breaking ties. Test evaluation follows checkpoint selection.


In [ ]:
names = {"real": "Real", "random": "Full-payload randomized",
         "header-only": "Header-only", "tls12_gcm_randomized": "TLS-GCM randomized"}
lines = ["| Condition | Accuracy (mean ± SD) | Macro F1 (mean ± SD) |",
         "|---|---:|---:|"]
for row in payload + [r for r in tls if r["condition"] != "real"]:
    lines.append(f"| {names[row['condition']]} | "
                 f"{100*float(row['accuracy_mean']):.2f}% ± {100*float(row['accuracy_std']):.2f}% | "
                 f"{float(row['macro_f1_mean']):.4f} ± {float(row['macro_f1_std']):.4f} |")
display(Markdown("\n".join(lines)))


## Capture-independence audit

**The available captures cannot support defensible six-class capture-independent
evaluation.** P2P has one recording, and Email's two recordings overlap and form
one acquisition group. Shared TCP tuples also link Chat and Streaming activities.
Final conservative candidate-group counts are 1/1/3/1/1/2 by class; their mutual
independence remains unverified. No independent split or training score was
created. Subdividing recordings cannot provide new acquisitions.


In [ ]:
feasibility = json.loads((ROOT / "results/capture_audit/feasibility.json").read_text())
assert feasibility["protocol"] == "STOP" and not feasibility["training_permitted"]
display(Markdown("| Class | Candidate groups |\n|---|---:|\n" +
                 "\n".join(f"| {name} | {count} |" for name, count in feasibility["group_counts"].items())))


## TLS 1.2 AES-GCM targeted ablation

The parser requires clear negotiated TLS 1.2 AES-GCM state, SYN-anchored exact
TCP offsets, ChangeCipherSpec and the expected protected Finished. Missing state,
gaps, conflicting overlaps, ambiguous boundaries and unsupported ciphers remain
untouched. TShark corroborates negotiated suites; Scapy/TShark checks corroborate
bounded packet offsets. This is structural evidence, without decryption or keys.

Randomize only confidently mapped ciphertext in retained packet prefixes,
including protected Finished, application-data and alert bodies. Preserve record
headers, explicit nonces, authentication tags, cleartext and unknown/unsupported
bytes. Rebuild every transformed graph; headers and untargeted packet graphs
remain identical. Modified inputs would fail TLS authentication by design.


In [ ]:
coverage = read_csv("results/tls_ciphertext_ablation/ciphertext_coverage.csv")
lines = ["| Scope | Affected samples | Targeted bytes / retained payload bytes |",
         "|---|---:|---:|"]
for row in coverage:
    if row["scope"] == "global" or (row["scope"] == "split" and row["id"] == "test"):
        lines.append(f"| {row['id']} | {row['samples_with_ciphertext']} / {row['samples']} | "
                     f"{int(row['ciphertext']):,} / {int(row['payload_bytes']):,} |")
display(Markdown("\n".join(lines)))

pairs = read_csv("results/tls_ciphertext_ablation/paired_differences.csv")
display(Markdown("| Seed | Real − TLS randomized accuracy (pp) | Macro F1 difference |\n|---|---:|---:|\n" +
    "\n".join(f"| {r['training_seed']} | {100*float(r['accuracy_difference']):+.2f} | "
              f"{float(r['macro_f1_difference']):+.4f} |" for r in pairs)))


## Interpretation and limitations

**Payload helps ≠ ciphertext helps.** The full-payload intervention reduced mean
performance. The targeted TLS intervention did not reduce performance in any of
the three paired seeds. This narrow observation does not establish statistical
equivalence or universal ciphertext irrelevance.

Only 17.97% of retained payload bytes and 85/250 test samples were targeted.
FileTransfer and P2P test inputs are untouched, and coverage is class-associated.
Much payload remains unknown or unsupported. Framing, cleartext, unsupported
encrypted regions, optimization and capture artifacts remain alternative
explanations. The retained layer is TCP application traffic; UDP OpenVPN tunnel
traffic is excluded and the acquisition interface remains unknown.

One split and one intervention seed were used. Three training seeds measure
training variability, not independent acquisitions or confidence intervals. The
test set was previously observed. Folder labels can include background traffic,
and long-flow exclusions can remove primary application flows. These are
exploratory flow-disjoint results; capture-independent generalization is unresolved.

## Reproduction

From the repository root, install the pinned Python 3.12 environment described
in the README, then run:

```sh
python -m unittest discover -s tests -v
python -m experiments.verify_results
```

See [the experiment guide](../experiments/README.md) for source capture hashes,
saved split validation, exact seeds/settings, input regeneration and optional
new training commands. Existing weights and raw captures are not distributed.
The notebook can be run from the repository root or `notebooks/`; it reads saved
CSV/JSON files only. Its saved outputs are intentionally empty.

## Attribution and references

This project adapts upstream components and includes AI-assisted implementation
and analysis tooling. It does not claim invention of TFE-GNN. See
[NOTICE](../NOTICE) and the preserved Apache-2.0 [LICENSE](../LICENSE).

- Zhang et al. (2023), TFE-GNN, WWW. [Official code and paper link](https://github.com/ViktorAxelsen/TFE-GNN).
- Draper-Gil et al. (2016), *Characterization of Encrypted and VPN Traffic Using
  Time-Related Features*, ICISSP. [ISCXVPN2016 dataset](https://www.unb.ca/cic/datasets/vpn.html).
- [RFC 5246](https://www.rfc-editor.org/rfc/rfc5246), [RFC 5288](https://www.rfc-editor.org/rfc/rfc5288),
  [RFC 5289](https://www.rfc-editor.org/rfc/rfc5289): TLS 1.2 and AES-GCM.
